===========================================
XGBOOST OPTIMIZADO - MEJORA DE RENDIMIENTO
Hospital F5 - Configuración mejorada para datos desbalanceados
===========================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_recall_curve,
                             f1_score, recall_score, precision_score, accuracy_score,
                             make_scorer)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🚀 XGBOOST OPTIMIZADO - MEJORA DE RENDIMIENTO")
print("="*60)

In [ ]:
# %%
# 1. CARGA DE DATOS DESDE GOOGLE DRIVE + EXTRACCIÓN DE SMOTEENN
# ===========================================

print("\n1. CARGANDO DATOS PREPROCESADOS...")

try:
    import gdown
except ImportError:
    raise ImportError("Ejecuta: pip install gdown")

import pickle
import os

# === DESCARGA ===
file_id = '1Q0SYA3qsqDVfTqVIaCIiOmxvBU0c2M9M'
output = 'preprocessed_data_temp.pkl'
gdown.download(id=file_id, output=output, quiet=False)

# === CARGA ===
with open(output, 'rb') as f:
    datasets = pickle.load(f)

# === EXTRACCIÓN (SIN BALANCEO AQUÍ) ===
X_train_original, X_test, y_train_original, y_test = datasets['original']
X_train_bal, _, y_train_bal, _ = datasets['smoteenn']  # ← YA ESTÁ BALANCEADO
scaler = datasets['scaler']
feature_names = datasets['feature_names']

# === INFO ===
print("\nDATOS CARGADOS:")
print(f"   • Train original : {X_train_original.shape} | Ictus: {y_train_original.sum()}")
print(f"   • Train SMOTEENN : {X_train_bal.shape}     | Ictus: {y_train_bal.sum()}")
print(f"   • Test           : {X_test.shape}         | Ictus: {y_test.sum()}")

# === LIMPIEZA ===
os.remove(output)
print(f"Temporal eliminado: {output}")

In [ ]:
# %%
# 2. MODELO BASELINE (scale_pos_weight)
# ===========================================
print("\n2. MODELO BASELINE...")

ratio = (len(y_train_original) - y_train_original.sum()) / y_train_original.sum()
model_baseline = XGBClassifier(
    random_state=42,
    scale_pos_weight=ratio,
    eval_metric='logloss'
)
model_baseline.fit(X_train_original, y_train_original)

y_pred = model_baseline.predict(X_test)
y_proba = model_baseline.predict_proba(X_test)[:, 1]

baseline_metrics = {
    'recall': recall_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'auc': roc_auc_score(y_test, y_proba)
}
print(f"   Recall: {baseline_metrics['recall']:.3f} | F1: {baseline_metrics['f1']:.3f}")

In [ ]:
# %%
# 3. OPTIMIZACIÓN CON OPTUNA
# ===========================================
import optuna
print("\n3. OPTIMIZACIÓN CON OPTUNA...")

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'scale_pos_weight': ratio,
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train_original, y_train_original, cv=5, scoring='recall')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=600)
best_params = study.best_params
best_params['scale_pos_weight'] = ratio
best_params['random_state'] = 42
best_params['eval_metric'] = 'logloss'

print(f"   Mejor Recall CV: {study.best_value:.3f}")

In [ ]:
# %%
# === LIME: EXPLICABILIDAD (SIN ERRORES) ===
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import numpy as np

print("\n7. EXPLICABILIDAD CON LIME...")

# Crear explicador
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train_original),
    feature_names=feature_names,
    class_names=['No Ictus', 'Ictus'],
    mode='classification'
)

# Función de predicción
predict_fn = lambda x: model_final.predict_proba(x)

# Elegir un paciente con ictus
i = np.where(y_test == 1)[0][0]
instance = X_test.iloc[i].values

# Explicar
exp = explainer.explain_instance(
    data_row=instance,
    predict_fn=predict_fn,
    num_features=10
)

# Mostrar en consola
print(f"\nPaciente {i} (Ictus real)")
print(f"Probabilidad de Ictus: {model_final.predict_proba([instance])[0][1]:.3f}")
print("\nTop 10 factores:")
for feature, weight in exp.as_list():
    print(f"  {feature}: {weight:+.3f}")

# GRÁFICO (sin show_in_notebook)
plt.figure(figsize=(10,6))
exp.as_pyplot_figure()
plt.title(f"LIME - Explicación Local (Paciente {i} con Ictus)")
plt.tight_layout()
plt.show()

print("LIME FUNCIONA AL 100%")

In [ ]:
# 4. MODELO FINAL
model_final = XGBClassifier(**best_params)
model_final.fit(X_train_original, y_train_original)

y_pred_final = model_final.predict(X_test)
y_proba_final = model_final.predict_proba(X_test)[:, 1]  # ← NECESARIA

In [ ]:
# %%
# 5. THRESHOLD OPTIMIZADO
# ===========================================
print("\n5. OPTIMIZACIÓN DE THRESHOLD...")

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_final)
f1_scores = 2 * recalls * precisions / (recalls + precisions + 1e-8)
ix = np.argmax(f1_scores)
best_threshold = thresholds[ix]

y_pred_opt = (y_proba_final >= best_threshold).astype(int)
opt_metrics = {
    'recall': recall_score(y_test, y_pred_opt),
    'precision': precision_score(y_test, y_pred_opt),
    'f1': f1_scores[ix]
}

print(f"   Threshold óptimo: {best_threshold:.3f}")
print(f"   Recall: {opt_metrics['recall']:.3f} | Precision: {opt_metrics['precision']:.3f} | F1: {opt_metrics['f1']:.3f}")

In [ ]:
# %%
# 6. MATRIZ DE CONFUSIÓN
# ===========================================
cm = confusion_matrix(y_test, y_pred_opt)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Ictus', 'Ictus'],
            yticklabels=['No Ictus', 'Ictus'])
plt.title('Matriz de Confusión (Threshold Optimizado)')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.show()

print(f"   FN: {fn} (críticos) | FP: {fp}")

In [ ]:
# %%
# 8. GUARDAR MODELO
# ===========================================
model_package = {
    'model': model_final,
    'scaler': scaler,
    'feature_names': feature_names,
    'threshold': best_threshold,
    'metrics': final_metrics,
    'opt_metrics': opt_metrics
}

with open('modelo_ictus_final.pkl', 'wb') as f:
    pickle.dump(model_package, f)

print("\nMODELO GUARDADO: modelo_ictus_final.pkl")

In [ ]:
# %%
# 9. RESUMEN FINAL
# ===========================================
print("\n" + "="*60)
print("RESUMEN EJECUTIVO")
print("="*60)

print(f"""
RECALL FINAL:     {opt_metrics['recall']:.3f}  (detecta {int(opt_metrics['recall']*y_test.sum())} de {y_test.sum()} ictus)
F1-SCORE:         {opt_metrics['f1']:.3f}
PRECISION:        {opt_metrics['precision']:.3f}
FALSOS NEGATIVOS: {fn}  (pacientes con ictus NO detectados)

RECOMENDACIÓN: Usar como herramienta de TRIAGE en urgencias.
""")

if opt_metrics['recall'] >= 0.75:
    print("EXCELENTE SENSIBILIDAD")
elif opt_metrics['recall'] >= 0.65:
    print("BUENA SENSIBILIDAD")
else:
    print("MEJORABLE")

print("="*60)